In [1]:
!pip install -q -U google-genai google-adk litellm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megacare-dev/agentic_rag_workshop/blob/main/th/4hr/agentic_rag_4hr_homework.ipynb)

# 📝 แบบฝึกหัด: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 ติดตั้ง Dependencies

In [2]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'scikit-learn']:
    _pip_install(_pkg)

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

  ⏭️  google-adk: skipped
  ⏭️  google-genai: skipped
  ⏭️  sentence-transformers: skipped
  ⏭️  qdrant-client: skipped
  ⏭️  langchain-text-splitters: skipped
  ⏭️  scikit-learn: skipped
✅ ติดตั้งเรียบร้อย!
CPU times: user 1.15 s, sys: 139 ms, total: 1.29 s
Wall time: 1.17 s


## 🎓 กรอกข้อมูลนักศึกษา

In [3]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = 'ชนม์นิภา เงาะลำดวน'
STUDENT_ID   = '6730401233'
PHONE        = ''
LINE_ID      = ''

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

✅ ชื่อ-นามสกุล: ชนม์นิภา เงาะลำดวน
✅ รหัสนักศึกษา: 6730401233
📱 เบอร์โทร: 
💬 LINE ID: 


## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [4]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

✅ สร้างข้อมูลเฉพาะสำหรับ 6730401233
📁 จำนวนไฟล์: 8 ไฟล์
🔍 Query เฉพาะตัว: "เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน"
  📄 doc_1.txt (136 ตัวอักษร)
  📄 doc_2.txt (146 ตัวอักษร)
  📄 doc_3.txt (150 ตัวอักษร)
  📄 doc_4.txt (152 ตัวอักษร)
  📄 doc_5.txt (164 ตัวอักษร)
  📄 doc_6.txt (121 ตัวอักษร)
  📄 doc_7.txt (141 ตัวอักษร)
  📄 doc_8.txt (150 ตัวอักษร)
CPU times: user 1.76 ms, sys: 152 µs, total: 1.91 ms
Wall time: 2.78 ms


---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน)

- รวมข้อความจากทุกไฟล์ใน `homework_data/`
- Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
- ค้นหาด้วย query: `MY_QUERY` (ที่สร้างจากรหัสนักศึกษา)
- เก็บลง Qdrant collection ชื่อ `f'hw_{STUDENT_ID}'`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ไหนมี similarity สูงสุด? (score ทศนิยม 4 ตำแหน่ง)
3. Top-3 ผลลัพธ์จาก Qdrant มี score เท่าไร?

In [5]:
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

files = sorted(glob.glob('homework_data/*.txt'))
all_text = '\n\n'.join(open(f, encoding='utf-8').read() for f in files)

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_text(all_text)
print(f'1) จำนวน chunks: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'   [{i}] ({len(c)} ตัวอักษร) {c}')

model = SentenceTransformer('intfloat/multilingual-e5-large')
embeddings = model.encode(['passage: ' + c for c in chunks], normalize_embeddings=True)
query_emb = model.encode(f'query: {MY_QUERY}', normalize_embeddings=True)

sims = cosine_similarity([query_emb], embeddings)[0]
best = int(np.argmax(sims))
print(f'\n2) chunk ที่คล้ายที่สุด: [{best}] score={sims[best]:.4f}\n   {chunks[best]}')

qdrant = QdrantClient(':memory:')
COLLECTION = f'hw_{STUDENT_ID}'

def build_collection(name, texts, vectors):
    if qdrant.collection_exists(name):
        qdrant.delete_collection(name)
    qdrant.create_collection(name, vectors_config=models.VectorParams(
        size=vectors.shape[1], distance=models.Distance.COSINE))
    qdrant.upsert(name, points=[models.PointStruct(id=i, vector=v.tolist(), payload={'text': t})
                                for i, (t, v) in enumerate(zip(texts, vectors))])

build_collection(COLLECTION, chunks, embeddings)
qdrant_results = qdrant.query_points(COLLECTION, query=query_emb.tolist(), limit=3).points
print('\n3) Top-3 จาก Qdrant:')
for r in qdrant_results:
    print(f'   [{r.id}] score={r.score:.4f} | {r.payload["text"]}')

# ✅ Self-check
assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
assert len(qdrant_results) == 3, '❌ ควรได้ top_k=3 จาก Qdrant'
print(f'✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

1) จำนวน chunks: 12
   [0] (136 ตัวอักษร) Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic
   [1] (146 ตัวอักษร) Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ
   [2] (139 ตัวอักษร) Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ
   [3] (26 ตัวอักษร) เช่น การจดจำภาพ การแปลภาษา
   [4] (142 ตัวอักษร) Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information
   [5] (29 ตัวอักษร) NLP และ Information Retrieval
   [6] (43 ตัวอักษร) การเรียนรู้ของเครื่อง หรือ Machine Learning
   [7] (142 ตัวอักษร) หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนม

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


2) chunk ที่คล้ายที่สุด: [5] score=0.8365
   NLP และ Information Retrieval

3) Top-3 จาก Qdrant:
   [5] score=0.8365 | NLP และ Information Retrieval
   [4] score=0.8260 | Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information
   [3] score=0.8220 | เช่น การจดจำภาพ การแปลภาษา
✅ Step 1 passed: 12 chunks, top score=0.8365


---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน)

- ตั้งค่า Gemini API Key (Colab Secrets)
- สร้าง **Custom Tool** อย่างน้อย 1 ตัว (ห้ามซ้ำกับ BMI ในคาบ)
- สร้าง **Agent** ด้วย Google ADK ที่ใช้ Tool ได้
- ทดลองคุย → แสดงว่า Agent เรียก Tool ได้จริง

**📝 รายงาน:**
1. Tool ของคุณทำอะไร? (อธิบาย 1-2 ประโยค)
2. แสดง output ที่ Agent เรียก Tool สำเร็จ
3. ทำไม docstring ถึงสำคัญ? (อธิบาย 1-2 ประโยค)

In [10]:
# ขั้นตอนที่ 2: Agent + Custom Tool

import os, asyncio, logging
from importlib.metadata import version
logging.getLogger('google_adk').setLevel(logging.CRITICAL)   # ซ่อน traceback ที่ ADK log เองตอนลองใหม่
import litellm
litellm.suppress_debug_info = True

# ─── ตั้งค่า LLM: ใช้ Gemini ผ่าน KKU IntelSphere (OpenAI-compatible) ───
# key เก็บใน Colab Secrets ชื่อ KKU_API_KEY — ห้ามพิมพ์ key ลงในโค้ด
# (เดิมใช้ Google AI Studio แต่ free tier ใช้ gemini-3.1-pro ไม่ได้ และรุ่น Flash ติด 503 บ่อย)
from google.colab import userdata
KKU_BASE  = 'https://gen.ai.kku.ac.th/api/v1'
KKU_KEY   = userdata.get('KKU_API_KEY')
KKU_MODEL = 'gemini-3.8-flash'

print(f'google-genai {version("google-genai")} | google-adk {version("google-adk")} | litellm {version("litellm")}')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.adk.models.lite_llm import LiteLlm
from google.genai import types as genai_types

LLM = LiteLlm(model=f'openai/{KKU_MODEL}', api_base=KKU_BASE, api_key=KKU_KEY, num_retries=3)
print(f'model: {KKU_MODEL} (via KKU IntelSphere)')

# ─── สร้าง Custom Tool (ห้ามใช้ BMI) ───
def calc_electricity_bill(kwh: float, rate_per_kwh: float) -> str:
    """คำนวณค่าไฟฟ้า (บาท) จากจำนวนหน่วยที่ใช้และราคาต่อหน่วย รวม VAT 7%

    Args:
        kwh: จำนวนหน่วยไฟฟ้าที่ใช้ (kWh)
        rate_per_kwh: ราคาค่าไฟต่อหน่วย (บาท/kWh)
    """
    base = kwh * rate_per_kwh
    total = base * 1.07
    return f'ค่าไฟก่อน VAT {base:,.2f} บาท, VAT 7% {base*0.07:,.2f} บาท, รวม {total:,.2f} บาท'

tool = FunctionTool(calc_electricity_bill)

# ─── สร้าง Agent ───
my_agent = LlmAgent(
    name='my_assistant',
    model=LLM,
    instruction='คุณเป็นผู้ช่วยคำนวณค่าไฟฟ้า เมื่อผู้ใช้ถามเรื่องค่าไฟ ให้เรียก tool calc_electricity_bill เสมอ ห้ามคำนวณเอง ตอบเป็นภาษาไทย',
    tools=[tool]
)

# ─── คุยกับ agent 1 รอบ (พิมพ์ trace ว่าเรียก tool อะไร ได้ผลอะไร) ───
async def _chat_once(agent, message, show_trace=True):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(app_name='homework', user_id='student')
    content = genai_types.Content(role='user', parts=[genai_types.Part(text=message)])
    response_text = ''
    async for event in runner.run_async(user_id='student', session_id=session.id, new_message=content):
        if show_trace:
            for call in event.get_function_calls():
                print(f'  🔧 CALL   {call.name}({dict(call.args)})')
            for resp in event.get_function_responses():
                print(f'  📥 RESULT {resp.name} → {str(resp.response)[:400]}')
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# ─── เรียก agent: ตัดทิ้งถ้าเกิน 90 วินาที + ลองใหม่เมื่อเจอ 503/429/timeout ───
async def chat_with_agent(agent, message, show_trace=True, retries=4, timeout=90):
    for attempt in range(retries):
        try:
            return await asyncio.wait_for(_chat_once(agent, message, show_trace), timeout=timeout)
        except Exception as e:
            err = 'timeout' if isinstance(e, asyncio.TimeoutError) else str(e)
            if any(k in err for k in ('503', '429', 'timeout')) and attempt < retries - 1:
                wait = 10 * (attempt + 1)
                print(f'  ⏳ {err[:40]}… รอ {wait}s แล้วลองใหม่ ({attempt + 1}/{retries - 1})')
                await asyncio.sleep(wait)
            else:
                raise

# ถามคำถามที่ต้องใช้ tool
answer = await chat_with_agent(my_agent, 'เดือนนี้ใช้ไฟ 235.5 หน่วย ค่าไฟหน่วยละ 4.18 บาท ต้องจ่ายเท่าไร')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent ไม่ตอบ'
print(f'✅ Step 2 passed!')

google-genai 2.34.0 | google-adk 2.15.1 | litellm 1.111.0
model: gemini-3.8-flash (via KKU IntelSphere)
  🔧 CALL   calc_electricity_bill({'kwh': 235.5, 'rate_per_kwh': 4.18})
  📥 RESULT calc_electricity_bill → {'result': 'ค่าไฟก่อน VAT 984.39 บาท, VAT 7% 68.91 บาท, รวม 1,053.30 บาท'}
🤖 Agent: ค่าไฟฟ้าสำหรับเดือนนี้มีรายละเอียดดังนี้ครับ

* **ค่าไฟก่อน VAT:** 984.39 บาท
* **ภาษีมูลค่าเพิ่ม (VAT 7%):** 68.91 บาท
* **ยอดรวมที่ต้องจ่ายทั้งหมด:** **1,053.30 บาท**
✅ Step 2 passed!


---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน)

- สร้าง **RAG Tool** ที่ค้นจาก Qdrant (ใช้ collection จากขั้นตอนที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร? (2-3 ประโยค)

In [11]:
# ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ

import json, time

# ─── A) สร้าง RAG Tool ───
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    q = model.encode(f'query: {query}', normalize_embeddings=True)
    hits = qdrant.query_points(COLLECTION, query=q.tolist(), limit=3).points
    return '\n'.join(f'[{h.score:.4f}] {h.payload["text"]}' for h in hits)

# ─── B) สร้าง RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model=LLM,
    instruction='คุณเป็น AI ที่ตอบคำถามจากฐานความรู้ ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ ตอบเป็นภาษาไทย',
    tools=[FunctionTool(search_knowledge)]
)

# ─── C) ถาม 3 คำถาม ───
questions = [
    MY_QUERY,  # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]

rag_answers = []
for q in questions:
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'🤖 {ans}')
    time.sleep(2)

# ─── D) LLM-as-Judge (ใช้ Gemini ผ่าน KKU IntelSphere ตัวเดียวกัน) ───
# 📋 Template สำหรับ judge (ใช้ตามนี้ได้เลย)
JUDGE_PROMPT = '''คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:
- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบ JSON: {{"score": 0, "reason": "..."}}
'''

def judge(prompt):
    """ให้ LLM ให้คะแนน แล้วดึง JSON ออกจากคำตอบ (กันกรณี model ใส่ ```json ครอบมา)"""
    resp = litellm.completion(
        model=f'openai/{KKU_MODEL}', api_base=KKU_BASE, api_key=KKU_KEY,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1, num_retries=3, timeout=60)
    text = resp.choices[0].message.content
    return json.loads(text[text.find('{'): text.rfind('}') + 1])

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
judge_scores = []
for qa in rag_answers:
    result = judge(JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer']))
    judge_scores.append(result['score'])
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')
    time.sleep(2)
print(f'  📈 คะแนนเฉลี่ย: {sum(judge_scores)/len(judge_scores):.2f}/5')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ ต้องตอบ 3 คำถาม'
print(f'\n✅ Step 3 passed: ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!')


❓ เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน
  🔧 CALL   search_knowledge({'query': 'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน semantic search vector search'})
  📥 RESULT search_knowledge → {'result': '[0.8652] Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว\n[0.8575] NLP และ Information Retrieval\n[0.8442] Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information'}
🤖 เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน (Semantic Search) ในทาง AI และ NLP นิยมใช้แนวทางและเทคนิคหลัก ๆ ดังนี้ครับ:

---

### 1. **การแปลงข้อมูลเป็น Embedding Vectors (Vector Embeddings)**
- นำข้อความ (หรือรูปภาพ/เสียง) ผ่านโมเดล AI (เช่น โมเดลตระกูล Transformers/Embedding Models) เพื่อแปลงข้อมูลเป็น **Vector** ตัวเลขหลายมิติที่บรรจุความหมายและบริบทของข้อมูลไว้ 
- ข้อความที่มีความหมายใกล้เคียงกันจะมีพิกัดเวกเตอร์อยู่ในบริเวณใกล้เคียงกัน แม้จะใช้

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [12]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')

👤 ชื่อ-นามสกุล: ชนม์นิภา เงาะลำดวน
🎓 รหัสนักศึกษา: 6730401233
📱 เบอร์โทร: 
💬 LINE ID: 
🔑 Verification Code: 87aa5a0c619b
📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.

📋 Checklist ก่อนส่ง:
  [ ] กรอกข้อมูลส่วนตัวครบถ้วน
  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน
  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน
  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge
  [ ] ทุก cell run แล้วมีผลลัพธ์


## 🔬 ระดับกลาง: Failure Analysis + Fix

In [13]:
EVAL = [
    ('Chunking มีกี่วิธี อะไรบ้าง',              ['Chunking', 'Fixed-size']),
    ('NLP ทำอะไรได้บ้าง',                       ['NLP', 'วิเคราะห์อารมณ์']),
    ('Deep Learning ใช้ทำงานอะไรได้บ้าง',          ['Deep Learning', 'การแปลภาษา']),
    ('Cosine Similarity นิยมใช้ในงานอะไร',       ['Cosine', 'Information Retrieval']),
    ('Machine Learning คืออะไร',               ['Machine Learning', 'เรียนรู้จากข้อมูล']),
    ('Transformer ใช้กลไกอะไร',                 ['Transformer', 'Attention']),
    ('Vector Database ช่วยอะไร',                ['Vector Database', 'ความหมายคล้ายกัน']),
    ('RAG ช่วยให้อ้างอิงแหล่งข้อมูลได้อย่างไร',        ['RAG', 'อ้างอิงแหล่งข้อมูล']),
]
OUT_OF_KB = ['Embedding คืออะไร?', 'Google ADK คืออะไร', 'Agent ต่างจาก Chatbot อย่างไร', 'Prompt Engineering คืออะไร']

def retrieval_metrics(collection, k=3):
    hit1 = hitk = 0
    for q, keys in EVAL:
        qv = model.encode(f'query: {q}', normalize_embeddings=True)
        hits = qdrant.query_points(collection, query=qv.tolist(), limit=k).points
        ok = [all(key in h.payload['text'] for key in keys) for h in hits]
        hit1 += ok[0]; hitk += any(ok)
        print(f'  {"✅" if ok[0] else "❌"} {q:<40} top1=[{hits[0].id}] {hits[0].score:.4f} {hits[0].payload["text"][:50]}')
    print(f'  → Hit@1 = {hit1}/{len(EVAL)} ({hit1/len(EVAL):.0%}) | Hit@{k} = {hitk}/{len(EVAL)} ({hitk/len(EVAL):.0%})')
    return hit1 / len(EVAL), hitk / len(EVAL)

print('📏 BEFORE (chunk_size=150, overlap=30):')
before_retrieval = retrieval_metrics(COLLECTION)

📏 BEFORE (chunk_size=150, overlap=30):
  ✅ Chunking มีกี่วิธี อะไรบ้าง              top1=[0] 0.9029 Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย
  ✅ NLP ทำอะไรได้บ้าง                        top1=[1] 0.9013 Natural Language Processing หรือ NLP คือสาขาที่ทำใ
  ❌ Deep Learning ใช้ทำงานอะไรได้บ้าง        top1=[2] 0.9085 Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช
  ❌ Cosine Similarity นิยมใช้ในงานอะไร       top1=[4] 0.9196 Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง V
  ✅ Machine Learning คืออะไร                 top1=[7] 0.9072 หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์
  ✅ Transformer ใช้กลไกอะไร                  top1=[8] 0.8695 Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใ
  ✅ Vector Database ช่วยอะไร                 top1=[9] 0.9081 Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก
  ❌ RAG ช่วยให้อ้างอิงแหล่งข้อมูลได้อย่างไร  top1=[10] 0.8736 Retrieval Augmented Generation หรือ RAG เป็นเทคนิค
  → Hit@1 = 5/8 (62%) | Hit@3 = 5/8 (62%)


In [14]:
FAILURE_PROBES = [
    'Embedding คืออะไร?',                      # F1: ไม่มีเอกสาร Embedding ในฐานความรู้
    'Deep Learning ใช้ทำงานอะไรได้บ้าง',          # F2: chunk ถูกตัดกลางประโยค
    'Machine Learning คืออะไร',               # F3: chunk เศษ (มีแค่ชื่อหัวข้อ) ติดอันดับสูง
    'Google ADK คืออะไร',                      # F4: นอกฐานความรู้ → agent ตอบจากความรู้ตัวเอง
]
probe_before = {}
for q in FAILURE_PROBES:
    print('=' * 60, f'\n❓ {q}')
    probe_before[q] = await chat_with_agent(rag_agent, q)
    print(f'🤖 {probe_before[q]}')
    time.sleep(2)

❓ Embedding คืออะไร?
  🔧 CALL   search_knowledge({'query': 'Embedding คืออะไร'})
  📥 RESULT search_knowledge → {'result': '[0.8631] Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic\n[0.8342] Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว\n[0.7966] เช่น การจดจำภาพ การแปลภาษา'}
  🔧 CALL   search_knowledge({'query': 'Embedding ความหมาย เวกเตอร์'})
  📥 RESULT search_knowledge → {'result': '[0.8465] Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic\n[0.8227] Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว\n[0.8212] เช่น การจดจำภาพ การแปลภาษา'}
🤖 **Embedding (หรือ Vector Embedding)** คือ กระบวนการแปลงข้อมูลรูปแบบต่าง ๆ เช่น ข้อความ (T

In [15]:
FAITH_PROMPT = '''คุณเป็นผู้ตรวจว่าคำตอบ "อิงจากเอกสารอ้างอิง" หรือไม่
เอกสารอ้างอิง:
{context}

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน faithfulness 1-5 (5 = ทุกข้อความมีในเอกสาร, 1 = แต่งขึ้นเองเกือบทั้งหมด หรือเอกสารไม่มีข้อมูลแต่ยังตอบ)
ตอบ JSON: {{"score": 0, "reason": "..."}}'''

q = 'Embedding คืออะไร?'
a = probe_before[q]
old = judge(JUDGE_PROMPT.format(question=q, answer=a))
new = judge(FAITH_PROMPT.format(context=search_knowledge(q), question=q, answer=a))
print(f'Judge เดิม (ไม่มี context): {old["score"]}/5 — {old["reason"]}')
print(f'Faithfulness (มี context):  {new["score"]}/5 — {new["reason"]}')

Judge เดิม (ไม่มี context): 5/5 — คำตอบมีความถูกต้อง ครบถ้วน และอธิบายได้อย่างชัดเจน ครอบคลุมตั้งแต่นิยามพื้นฐาน (การแปลงข้อมูลเป็นเวกเตอร์ตัวเลขหลายมิติ) หลักการสำคัญ (การจับความหมายเชิงบริบท หรือ Semantic Meaning) ไปจนถึงการประยุกต์ใช้งานจริงร่วมกับเทคโนโลยีปัจจุบัน เช่น Vector Database และ RAG มีการจัดโครงสร้างเนื้อหาที่อ่านเข้าใจง่าย
Faithfulness (มี context):  2/5 — เอกสารอ้างอิงไม่มีการระบุนิยามว่า Embedding คืออะไรอย่างชัดเจน คำตอบหลักที่เป็นการอธิบายความหมายของ Embedding (การแปลงข้อมูลเป็นเวกเตอร์ตัวเลขในปริภูมิหลายมิติ) จึงเป็นการนำความรู้ภายนอกมาตอบ มีเพียงข้อมูลเรื่อง Vector Database และ Chunking ในข้อ 2 และ 3 เท่านั้นที่สอดคล้องกับเอกสารอ้างอิง


In [16]:
# เอกสารยาวสุด 164 ตัวอักษร → chunk_size=200 ทำให้ 1 เอกสาร = 1 chunk
# (ถ้าใช้ 300 splitter จะรวม 2 เอกสารเป็น chunk เดียว เหลือแค่ 5 chunks)
splitter_v2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0, separators=['\n\n', ' ', ''])
chunks_v2 = splitter_v2.split_text(all_text)
emb_v2 = model.encode(['passage: ' + c for c in chunks_v2], normalize_embeddings=True)
COLLECTION_V2 = f'hw_{STUDENT_ID}_v2'
build_collection(COLLECTION_V2, chunks_v2, emb_v2)
print(f'chunks: {len(chunks)} → {len(chunks_v2)} | สั้นที่สุด: {min(map(len, chunks))} → {min(map(len, chunks_v2))} ตัวอักษร')

print('\n📏 AFTER FIX-1 (chunk ตามย่อหน้า):')
after_retrieval = retrieval_metrics(COLLECTION_V2)

chunks: 12 → 8 | สั้นที่สุด: 26 → 121 ตัวอักษร

📏 AFTER FIX-1 (chunk ตามย่อหน้า):
  ✅ Chunking มีกี่วิธี อะไรบ้าง              top1=[0] 0.9029 Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย
  ✅ NLP ทำอะไรได้บ้าง                        top1=[1] 0.9013 Natural Language Processing หรือ NLP คือสาขาที่ทำใ
  ✅ Deep Learning ใช้ทำงานอะไรได้บ้าง        top1=[2] 0.9116 Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช
  ✅ Cosine Similarity นิยมใช้ในงานอะไร       top1=[3] 0.9225 Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง V
  ✅ Machine Learning คืออะไร                 top1=[4] 0.9067 การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสา
  ✅ Transformer ใช้กลไกอะไร                  top1=[5] 0.8695 Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใ
  ✅ Vector Database ช่วยอะไร                 top1=[6] 0.9081 Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก
  ✅ RAG ช่วยให้อ้างอิงแหล่งข้อมูลได้อย่างไร  top1=[7] 0.8880 Retrieval Augmented Generation หรือ RAG เป็นเทคนิค
  → Hit@1 = 8/8 (100%)

In [17]:
def search_knowledge_v2(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ (Qdrant) — ต้องเรียกก่อนตอบทุกครั้ง
    ผลลัพธ์คือข้อความจากเอกสารพร้อม score ความคล้าย (0-1)

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    q = model.encode(f'query: {query}', normalize_embeddings=True)
    hits = qdrant.query_points(COLLECTION_V2, query=q.tolist(), limit=3).points
    return '\n'.join(f'[{h.score:.4f}] {h.payload["text"]}' for h in hits)

rag_agent_v2 = LlmAgent(
    name='rag_assistant_v2',
    model=LLM,
    instruction=('คุณเป็น AI ที่ตอบคำถามจากฐานความรู้เท่านั้น ใช้ tool search_knowledge_v2 ก่อนตอบเสมอ\n'
                 '- ตอบโดยใช้เฉพาะข้อความที่อยู่ในผลค้นหา ห้ามเติมความรู้ของตัวเอง\n'
                 '- ถ้าผลค้นหาไม่ได้อธิบายสิ่งที่ถามโดยตรง (แค่มีคำนั้นปรากฏ ไม่นับ) '
                 'ให้ตอบว่า "ไม่พบข้อมูลนี้ในฐานความรู้" แล้วบอกว่าพบหัวข้อใกล้เคียงอะไรบ้าง\n'
                 '- ตอบเป็นภาษาไทย'),
    tools=[FunctionTool(search_knowledge_v2)]
)

async def answer_metrics(agent, label):
    refused = answered = 0
    for q in OUT_OF_KB:
        a = await chat_with_agent(agent, q, show_trace=False); time.sleep(2)
        refused += 'ไม่พบ' in a
        print(f'  {"✅ ปฏิเสธ" if "ไม่พบ" in a else "❌ ตอบเอง"} | {q} → {a[:80]}')
    for q, _ in EVAL:
        a = await chat_with_agent(agent, q, show_trace=False); time.sleep(2)
        answered += 'ไม่พบ' not in a
    print(f'  → [{label}] ปฏิเสธคำถามนอกฐานความรู้ {refused}/{len(OUT_OF_KB)} | ตอบคำถามในฐานความรู้ได้ {answered}/{len(EVAL)}')
    return refused / len(OUT_OF_KB), answered / len(EVAL)

print('📏 BEFORE (rag_agent เดิม):')
before_answer = await answer_metrics(rag_agent, 'before')
print('\n📏 AFTER (rag_agent_v2):')
after_answer = await answer_metrics(rag_agent_v2, 'after')

📏 BEFORE (rag_agent เดิม):
  ❌ ตอบเอง | Embedding คืออะไร? → **Embedding (หรือ Vector Embedding)** คือ การแปลงข้อมูลประเภทต่าง ๆ เช่น ข้อความ
  ❌ ตอบเอง | Google ADK คืออะไร → จากการค้นหาในฐานความรู้ไม่มีข้อมูลนี้โดยตรง แต่ตามข้อมูลทั่วไปของ Google คำว่า *
  ❌ ตอบเอง | Agent ต่างจาก Chatbot อย่างไร → ความแตกต่างสำคัญระหว่าง **Chatbot** และ **AI Agent** สรุปได้ดังนี้ครับ:

---

##
  ❌ ตอบเอง | Prompt Engineering คืออะไร → **Prompt Engineering (วิศวกรรมคำสั่ง)** คือ กระบวนการออกแบบ ปรับแต่ง และเรียบเรี
  → [before] ปฏิเสธคำถามนอกฐานความรู้ 0/4 | ตอบคำถามในฐานความรู้ได้ 8/8

📏 AFTER (rag_agent_v2):
  ✅ ปฏิเสธ | Embedding คืออะไร? → ไม่พบข้อมูลนี้ในฐานความรู้

หัวข้อใกล้เคียงที่พบในฐานความรู้ ได้แก่:
- **Chunkin
  ✅ ปฏิเสธ | Google ADK คืออะไร → ไม่พบข้อมูลนี้ในฐานความรู้

หัวข้อใกล้เคียงที่พบในฐานความรู้ ได้แก่:
- **Retriev
  ✅ ปฏิเสธ | Agent ต่างจาก Chatbot อย่างไร → ไม่พบข้อมูลนี้ในฐานความรู้

หัวข้อใกล้เคียงที่พบในฐานความรู้ ได้แก่:
- **Transfo
  ✅ ปฏิเสธ | Prompt Engineering คืออะไร →

In [18]:
probe_after = {}
for q in FAILURE_PROBES:
    print('=' * 60, f'\n❓ {q}')
    probe_after[q] = await chat_with_agent(rag_agent_v2, q)
    print(f'🤖 {probe_after[q]}')
    time.sleep(2)

q = 'Embedding คืออะไร?'
new = judge(FAITH_PROMPT.format(context=search_knowledge_v2(q), question=q, answer=probe_after[q]))
print(f'\nFaithfulness หลังแก้: {new["score"]}/5 — {new["reason"]}')

❓ Embedding คืออะไร?
  🔧 CALL   search_knowledge_v2({'query': 'Embedding คืออะไร'})
  📥 RESULT search_knowledge_v2 → {'result': '[0.8631] Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic\n[0.8342] Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว\n[0.7901] Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น N
🤖 ไม่พบข้อมูลนี้ในฐานความรู้

อย่างไรก็ตาม ในฐานความรู้พบหัวข้อใกล้เคียง ดังนี้:
- **Chunking**: กระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยสำหรับการสร้าง Embedding
- **Vector Database**: ฐานข้อมูลที่จัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector
- **Deep Learning**: เทคนิค Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้นในการประมวลผลข้อมูลที่ซับซ้อน
❓ Deep Learning ใช้ทำงานอะไรได้บ้าง
  🔧 CALL   search_knowledge_v2({'query': 'Deep Learning ใช้ทำงานอะไรได้บ้าง'})
  📥 RESULT sea

In [20]:
import pandas as pd
summary = pd.DataFrame({
    'Metric': ['Retrieval Hit@1 (8 คำถาม)',
               'Retrieval Hit@3 (8 คำถาม)',
               'ปฏิเสธคำถามนอกฐานความรู้ (4 คำถาม)',
               'ตอบคำถามในฐานความรู้ได้ (ไม่ปฏิเสธเกิน · 8 คำถาม)'],
    'Before': [before_retrieval[0], before_retrieval[1], before_answer[0], before_answer[1]],
    'After':  [after_retrieval[0],  after_retrieval[1],  after_answer[0],  after_answer[1]],
})
summary['Δ'] = summary['After'] - summary['Before']
summary.style.format({'Before': '{:.0%}', 'After': '{:.0%}', 'Δ': '{:+.0%}'}).hide(axis='index')

Metric,Before,After,Δ
Retrieval Hit@1 (8 คำถาม),62%,100%,+38%
Retrieval Hit@3 (8 คำถาม),62%,100%,+38%
ปฏิเสธคำถามนอกฐานความรู้ (4 คำถาม),0%,100%,+100%
ตอบคำถามในฐานความรู้ได้ (ไม่ปฏิเสธเกิน · 8 คำถาม),100%,88%,-12%


In [21]:
for q, _ in EVAL:
    a = await chat_with_agent(rag_agent_v2, q, show_trace=False); time.sleep(2)
    if 'ไม่พบ' in a:
        print(f'❌ ปฏิเสธเกิน: {q}\n   → {a[:150]}')
        print('   ผลค้นหา:', search_knowledge_v2(q)[:200])

❌ ปฏิเสธเกิน: RAG ช่วยให้อ้างอิงแหล่งข้อมูลได้อย่างไร
   → ไม่พบข้อมูลนี้ในฐานความรู้

ในฐานความรู้มีเพียงการระบุว่า Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความ
   ผลค้นหา: [0.8880] Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้
[0.7731] Vector Database เป็นฐานข้อมูลที
